# Raspa3から取得したデータを読み込む

In [ ]:
### --- import block --- ###
import numpy as np
import pprint
from c02_raspa_data_parsers import RaspaForceField


### --- main block --- ###
json = rf"C:\Users\y_7up\Documents\projects\mi_dev\04_mof\input\02_GCMC_input\11_mc_adsorption_of_co2_in_cu_btc\force_field.json"

ff = RaspaForceField(json)

# 概要を確認
# ff.summary()

framework_atoms = ff.framework_atom_types()
print(f"framework_atoms:{framework_atoms}") # framework_atoms:['Cu1', 'O1', 'C1', 'C2', 'C3', 'H1']

# 吸着質()
adsorbate_atoms = ff.adsorbate_atom_types()
print(f"adsorbate_atoms:{adsorbate_atoms}") # adsorbate_atoms:['C_co2', 'O_co2']
print()

# 吸着質-骨格原子の全epx, sigの組み合わせを取得
# より洗練された設計：行列を作る（行列演算の方が後々便利）
# Ex. energy = 4 * eps_matrix * (...)

# 行列の次元を決定
n_ads = len(adsorbate_atoms)
n_frm = len(framework_atoms)
d_mat = n_ads + n_frm

# key_mapも自動生成でなければ意味がない
type_list = adsorbate_atoms + framework_atoms
type_map = dict()

for i, type_id in enumerate(type_list):
    type_map[type_id] = i

# type_mapの中身確認
pprint.pprint(type_map)
print()

eps_matrix = np.zeros((d_mat, d_mat))
sig_matrix = np.zeros((d_mat, d_mat))

for atom_i in type_list:
    for atom_j in type_list:
        i = type_map[atom_i]
        j = type_map[atom_j]
        eps_matrix[i, j], sig_matrix[i, j] = ff.mix_lj(atom_i, atom_j) # データの実体はffに入っている

print(eps_matrix.shape)

# pprint.pprint(eps_matrix)


# 実務では以下のように取り出す
# クロス積のみの場合：
for atom_i in adsorbate_atoms:
    for atom_j in framework_atoms:
        print(f"eps[{atom_i}, {atom_j}]:{eps_matrix[type_map[atom_i], type_map[atom_j]]}")

print()

# 全原子の組み合わせの場合:
for atom_i in type_list:
    for atom_j in type_list:
        print(f"eps[{atom_i}, {atom_j}]:{eps_matrix[type_map[atom_i], type_map[atom_j]]}")


framework_atoms:['Cu1', 'O1', 'C1', 'C2', 'C3', 'H1']
adsorbate_atoms:['C_co2', 'O_co2']

{'C1': 4, 'C2': 5, 'C3': 6, 'C_co2': 0, 'Cu1': 2, 'H1': 7, 'O1': 3, 'O_co2': 1}

(8, 8)
eps[C_co2, Cu1]:72.15613405787589
eps[C_co2, O1]:315.6777554658163
eps[C_co2, C1]:314.68671816734826
eps[C_co2, C2]:314.68671816734826
eps[C_co2, C3]:314.68671816734826
eps[C_co2, H1]:125.80840347109005
eps[O_co2, Cu1]:122.07171668173834
eps[O_co2, O1]:534.0547415835958
eps[O_co2, C1]:532.3781325759321
eps[O_co2, C2]:532.3781325759321
eps[O_co2, C3]:532.3781325759321
eps[O_co2, H1]:212.83911597018874

eps[C_co2, C_co2]:248.876809544594
eps[C_co2, O_co2]:421.04278146351925
eps[C_co2, Cu1]:72.15613405787589
eps[C_co2, O1]:315.6777554658163
eps[C_co2, C1]:314.68671816734826
eps[C_co2, C2]:314.68671816734826
eps[C_co2, C3]:314.68671816734826
eps[C_co2, H1]:125.80840347109005
eps[O_co2, C_co2]:421.04278146351925
eps[O_co2, O_co2]:712.3083269466781
eps[O_co2, Cu1]:122.07171668173834
eps[O_co2, O1]:534.0547415835958
e